# Simulate RI Logit

## Packages

In [1]:
import numpy as np

from scipy.optimize import minimize
from src.simulation.simulate_logit import SimulateRILogit

## Global Parameters

#TODO find a simulation that gives rise to marg all interiors and then estimate the logit MLE. 

In [2]:
J = 3 # Number of products
N = 3 # Number of states per products
llambda = 0.1 # Information Cost
I = 1 # Number of Individuals
n_sim = 10000 # Number of simulation for each individuals
state = (0, 0, 0)

## Simulate Individuals

for now we just pick individuals payoff and prior matrix  matrix. In the future to assess the performance of our estimators we will have to get payoff and prior matrix as function of individual characteristics and parameters $\theta \in \Theta$ to be estimated.

In [3]:
list_u_mat = [np.random.beta(a = 0.5, b = 0.8, size=(N, J)) for _ in range(I)] # Draws payoff matrix for each individuals
list_ppi = [np.array([np.random.dirichlet(np.ones(N)) for _ in range(J)]).T for _ in range(I)] # Draws prior matrix for each individuals

In [43]:
list_positive_u_mat = [
    np.array(
        [
            [0.1671905, 0.00128109, 0.14230163],
            [0.99809904, 0.00228897, 0.10311815],
            [0.01250213, 0.32587882, 0.22736151],
        ]
    )
]

In [41]:
list_u_mat

[array([[0.1671905 , 0.00128109, 0.14230163],
        [0.99809904, 0.00228897, 0.10311815],
        [0.01250213, 0.32587882, 0.22736151]])]

## Simulate Logit Choice

Using Blahut–Arimoto solver

In [51]:
list_classes_BA = [
    SimulateRILogit(u_mat=u_mat, ppi=ppi, llambda=llambda)
    for u_mat, ppi in zip(list_u_mat, list_ppi)
]
list_indexes_BA = [cl.get_states().index(state) for cl in list_classes_BA]
list_choices_BA = [cl.simulate(n_sim=n_sim, states=state) for cl in list_classes_BA]
list_dist_BA = [cl.get_logit()[index] for cl, index in zip(list_classes_BA, list_indexes_BA)]
list_marg_BA = [cl.get_marg() for cl in list_classes_BA]

Blahut–Arimoto Solver:   0%|          | 18/10000 [00:00<00:03, 2956.97iter/s, Error=[7.89732622e-13]]


In [39]:
[cl.get_logit() for cl in list_classes_BA]

Blahut–Arimoto Solver:   0%|          | 18/10000 [00:00<00:07, 1338.89iter/s, Error=[7.89732622e-13]]


[array([[4.27087702e-01, 1.43729858e-01, 4.29182440e-01],
        [4.96114156e-01, 1.66959659e-01, 3.36926185e-01],
        [2.71071397e-01, 9.12249481e-02, 6.37703655e-01],
        [4.26466789e-01, 1.44974731e-01, 4.28558480e-01],
        [4.95276514e-01, 1.68366169e-01, 3.36357317e-01],
        [2.70821135e-01, 9.20639597e-02, 6.37114905e-01],
        [9.39019452e-02, 8.11735549e-01, 9.43625060e-02],
        [9.68651313e-02, 8.37350817e-01, 6.57840514e-02],
        [8.33539483e-02, 7.20553369e-01, 1.96092682e-01],
        [9.99669754e-01, 8.28507648e-05, 2.47395314e-04],
        [9.99749936e-01, 8.28574101e-05, 1.67207044e-04],
        [9.99338204e-01, 8.28232866e-05, 5.78972240e-04],
        [9.99668915e-01, 8.36899529e-05, 2.47395106e-04],
        [9.99749096e-01, 8.36966655e-05, 1.67206904e-04],
        [9.99337366e-01, 8.36621964e-05, 5.78971755e-04],
        [9.97629282e-01, 2.12382733e-03, 2.46890344e-04],
        [9.97709137e-01, 2.12399733e-03, 1.66865723e-04],
        [9.972

In [40]:
epsilon = 1e-3
positive_states = [np.all(cl.get_logit() >= epsilon, axis=1).tolist() for cl in list_classes_BA]
[
    [num for num, flag in zip(cl.get_states(), positive_state) if flag]
    for cl, positive_state in zip(list_classes_BA, positive_states)
]

Blahut–Arimoto Solver:   0%|          | 18/10000 [00:00<00:09, 1011.52iter/s, Error=[7.89732622e-13]]


[[(0, 0, 0),
  (0, 0, 1),
  (0, 0, 2),
  (0, 1, 0),
  (0, 1, 1),
  (0, 1, 2),
  (0, 2, 0),
  (0, 2, 1),
  (0, 2, 2),
  (2, 0, 0),
  (2, 0, 1),
  (2, 0, 2),
  (2, 1, 0),
  (2, 1, 1),
  (2, 1, 2),
  (2, 2, 0),
  (2, 2, 1),
  (2, 2, 2)]]

In [48]:
list_classes_SQP = [
    SimulateRILogit(u_mat=u_mat, ppi=ppi, llambda=llambda, method="SQP")
    for u_mat, ppi in zip(list_u_mat, list_ppi)
]
list_indexes_SQP = [cl.get_states().index(state) for cl in list_classes_SQP]
list_choices_SQP = [cl.simulate(n_sim=n_sim, states=state) for cl in list_classes_SQP]
list_dist_SQP = [cl.get_logit()[index] for cl, index in zip(list_classes_SQP, list_indexes_SQP)]

SQP Solver:   0%|          | 1/10000 [00:00<01:03, 156.52iter/s, Error=0.00237]


ValueError: Invalid dimensions (0,).

In [43]:
for list in list_dist_SQP:
    print(list)

[1.62925109e-04 9.99837075e-01 0.00000000e+00]


## Estimation

In [46]:
list_choices_BA[0][1]*n_sim

array([[4198., 1476., 4326.]])

In [52]:
list_marg_BA

[array([[0.24647369],
        [0.43584885],
        [0.31767746]])]

In [ ]:
n_j = list_choices_BA[0][1] * n_sim  # Number of times each product is chosen

# Initial values for B and q
u_init = np.ones(J)  # Start with all u_j = 1
q_init = np.full(J, 1 / J)  # Start with uniform q_j


# Constraint: sum(q) = 1
def constraint_q(q):
    return np.sum(q) - 1


# Log-likelihood function
def log_likelihood(params):
    u = params[:J]  # First J elements are u_j
    q = params[J:]  # Last J elements are q_j

    b = np.exp(u)

    if np.any(q <= 0) or np.any(q >= 1):
        return np.inf  # Enforce positivity of B and valid q

    denominator = np.sum(b * q)
    log_likelihood_value = np.sum(n_j * np.log(b * q / denominator))

    return -log_likelihood_value  # Negative for minimization


# Initial parameter guess
params_init = np.concatenate([u_init, q_init])

# Bounds: B_j > 0, 0 < q_j < 1
bounds = [(1e-6, None)] * J + [(1e-6, 1 - 1e-6)] * J

# Constraints for q sum
constraints = [{"type": "eq", "fun": constraint_q}]

# Optimization
result = minimize(
    log_likelihood, params_init, bounds=bounds, constraints=constraints, method="SLSQP"
)

# Extract results
u_est = result.x[:J]
q_est = result.x[J:]

print("Estimated B:", u_est)
print("Estimated q:", q_est)

Estimated B: [0.0579703  0.02636532 0.06066121]
Estimated q: [0.35765293 0.12978632 0.36756393]


/Users/maximecoulet/miniconda3/envs/RIProject/lib/python3.12/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Users/maximecoulet/miniconda3/envs/RIProject/lib/python3.12/site-packages/scipy/optimize/_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
/Users/maximecoulet/miniconda3/envs/RIProject/lib/python3.12/site-packages/scipy/optimize/_slsqp_py.py:493: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])


In [4]:
import numpy as np
from scipy.stats import multivariate_normal, expon
from scipy.optimize import minimize

# Seed for reproducibility
np.random.seed(42)

# Simulation parameters
N = 5000  # Number of individuals
J = 3  # Number of inside choices
R = 100  # Simulated shocks per individual

# True parameters
mu = np.array([-1, 0.5])  # Mean vector
Sigma = np.array([[0.5, 0], [0, 0.9]])  # Covariance matrix

# Draw random coefficients
beta_i = np.random.multivariate_normal(mu, Sigma, size=N).T

# Generate characteristics
p_ij = expon(scale=1).rvs(size=(N, J))
x_ij = np.array([2.2, 3.8, 1.4]) + expon(scale=2.5).rvs(size=(N, J))

# Compute utilities
v_ij = beta_i[0, :] * p_ij + beta_i[1, :] * x_ij
s_ij = np.exp(v_ij)
s_ij = np.hstack([s_ij, np.ones((N, 1))])  # Adding outside good
s_ij /= s_ij.sum(axis=1, keepdims=True)  # Normalize

# Market shares
sigma_j = s_ij.mean(axis=0)

# Simulate choices
y_ij = np.zeros((N, J + 1))
for i in range(N):
    y_ij[i, np.random.choice(J + 1, p=s_ij[i, :])] = 1

# Repeat for shocks
p_irj = np.tile(p_ij, (R, 1))
x_irj = np.tile(x_ij, (R, 1))

# Initial parameter guess
theta_init = np.array([-0.25, 0.25, -0.7, -0.7])


def A(a, b):
    """Construct diagonal matrix"""
    return np.diag([np.exp(a), np.exp(b)])


def T(theta):
    """Compute covariance transformation"""
    return A(theta[2], theta[3]) @ A(theta[2], theta[3]).T


# Simulated Likelihood Function
o_ir = np.random.multivariate_normal([0, 0], np.eye(2), size=N * R).T


def s_ij_sim(theta):
    """Compute simulated choice probabilities"""
    beta_ir = theta[:2].reshape(-1, 1) + T(theta) @ o_ir
    utilities = beta_ir[0, :] * p_irj + beta_ir[1, :] * x_irj
    s_irj = np.exp(utilities)
    s_irj = np.hstack([s_irj, np.ones((N * R, 1))])
    s_irj /= s_irj.sum(axis=1, keepdims=True)
    return s_irj.reshape(R, N, J + 1).mean(axis=0)


def log_likelihood_msl(theta):
    """Compute negative log-likelihood for MSL"""
    return -np.sum(np.log((y_ij * s_ij_sim(theta)).sum(axis=1)))


# Maximum Simulated Likelihood Estimation
result_msl = minimize(
    log_likelihood_msl, theta_init, method="Nelder-Mead", options={"maxiter": 500}
)
theta_msl = result_msl.x
print("MSL Estimate:", theta_msl)


# Importance Sampling
def s_irj_is(theta, o_ir):
    """Compute choice probabilities under importance sampling"""
    beta_ir = o_ir
    utilities = beta_ir[0, :] * p_irj + beta_ir[1, :] * x_irj
    s_irj = np.exp(utilities)
    s_irj = np.hstack([s_irj, np.ones((N * R, 1))])
    s_irj /= s_irj.sum(axis=1, keepdims=True)
    proposal_density = multivariate_normal(theta[:2], T(theta)).pdf(o_ir.T)
    return s_irj, proposal_density


def s_ij_is(theta, s_irj, proposal_density, o_ir):
    """Compute importance-weighted choice probabilities"""
    true_density = multivariate_normal(theta[:2], T(theta)).pdf(o_ir.T)
    weighted_s_irj = s_irj * true_density[:, None] / proposal_density[:, None]
    return weighted_s_irj.reshape(R, N, J + 1).mean(axis=0)


def log_likelihood_is(theta):
    """Compute negative log-likelihood for IS"""
    s_irj, proposal_density = s_irj_is(theta, o_ir)
    return -np.sum(
        np.log((y_ij * s_ij_is(theta, s_irj, proposal_density, o_ir)).sum(axis=1))
    )


# Importance Sampling Estimation
result_is = minimize(
    log_likelihood_is, theta_init, method="Nelder-Mead", options={"maxiter": 500}
)
theta_is = result_is.x
print("IS Estimate:", theta_is)


# Iterative Importance Sampling
def importance_sampling_iterative(proposal_theta, max_iter=5):
    """Iteratively update the proposal distribution"""
    theta = proposal_theta
    o_ir_fixed = np.random.multivariate_normal([0, 0], np.eye(2), size=N * R).T

    for i in range(max_iter):
        print(f"Iteration {i+1}, Current theta:", theta)
        o_ir = theta[:2].reshape(-1, 1) + A(theta[2], theta[3]) @ o_ir_fixed
        s_irj, proposal_density = s_irj_is(theta, o_ir)

        result = minimize(
            lambda t: log_likelihood_is(t),
            theta,
            method="Nelder-Mead",
            options={"maxiter": 500},
        )
        theta = result.x

    return theta


# Run iterative IS
theta_iter_is = importance_sampling_iterative(theta_init, max_iter=5)
print("Final Iterative IS Estimate:", theta_iter_is)

ValueError: operands could not be broadcast together with shapes (5000,) (5000,3) 